# Resistance Model Inference

In [1]:
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from axiom.config import DATA_DIR

## Load the data

In [2]:
TICKER = "SPY"

SAVE_FN = f"{TICKER}_{datetime.now().strftime('%Y-%m-%d')}.csv"
# SAVE_FN = "SPY_2024-07-14.csv"

SAVE_FP = DATA_DIR.joinpath("daily", SAVE_FN)

In [3]:
# GET the data

from axiom.mdata.equity import get_daily_price_history, CandleList

data: CandleList = await get_daily_price_history(TICKER)

# Convert to DataFrame
df = pd.DataFrame.from_records([candle.dict() for candle in data.candles])
df.to_csv(SAVE_FP, index=False)

In [4]:
# Load the data
df = pd.read_csv(SAVE_FP)

# Filter columns
columns = ["datetime", "high", "low"]
df = df[columns]

# Convert int to datetime
df["datetime"] = pd.to_datetime(df["datetime"], unit="ms")

# Add a new column for the day of the week, 0=Monday, 6=Sunday
df["day"] = df["datetime"].dt.dayofweek


# Round the high and low
def round_to_tick(value):
    tick = value // 100 or 1
    round_to = tick / 10
    return round(value / round_to) * round_to


# Apply the custom rounding to 'high' and 'low' columns
df["high"] = df["high"].apply(round_to_tick)
df["low"] = df["low"].apply(round_to_tick)

df.tail(10)

,datetime,high,low,day
5026,2024-07-08 05:00:00,556.5,554.0,0
5027,2024-07-09 05:00:00,557.0,555.5,1
5028,2024-07-10 05:00:00,561.5,557.0,2
5029,2024-07-11 05:00:00,562.5,556.0,3
5030,2024-07-12 05:00:00,563.5,557.0,4
5031,2024-07-15 05:00:00,565.0,559.5,0
5032,2024-07-16 05:00:00,565.0,562.0,1
5033,2024-07-17 05:00:00,560.5,556.5,2
5034,2024-07-18 05:00:00,559.5,550.5,3
5035,2024-07-19 05:00:00,554.0,548.0,4


In [21]:
# In the last 10 rows, get the index of the last "4" day
last_4_day_index = df[df["day"] == 4].index[-1]

# Get the highs and lows of the last 5 days ending on the last "4" day
last_5_days = df.loc[last_4_day_index - 4 : last_4_day_index]
last_5_days

,datetime,high,low,day
5031,2024-07-15 05:00:00,565.0,559.5,0
5032,2024-07-16 05:00:00,565.0,562.0,1
5033,2024-07-17 05:00:00,560.5,556.5,2
5034,2024-07-18 05:00:00,559.5,550.5,3
5035,2024-07-19 05:00:00,554.0,548.0,4


In [22]:
highs = last_5_days["high"].values
lows = last_5_days["low"].values

## Load the Model

In [23]:
from axiom.models import WeeklyResistanceModel

In [24]:
MODEL_HIGH_FP = DATA_DIR.joinpath("models", f"weekly_resistance_model_high_SPY.json")
MODEL_LOW_FP = DATA_DIR.joinpath("models", f"weekly_resistance_model_low_SPY.json")

# Load the model
model = WeeklyResistanceModel.load("SPY", MODEL_HIGH_FP, MODEL_LOW_FP)

In [27]:
high_resistance, low_resistance = model.predict(highs, lows)

print(f"{TICKER} friday close price: {df.iloc[last_4_day_index]['high']}")
print(f"High Resistance: {high_resistance}")
print(f"Low Resistance: {low_resistance}")

SPY friday close price: 554.0
High Resistance: 542.6912231445312
Low Resistance: 541.069091796875
